# faissight in a notebook

Build a small IVF index, open the faissight UI inline, and ask it *why* retrieval missed
neighbours, from the UI or from Python.

```bash
pip install "faissight[faiss-cpu]"
```

In [ ]:
import faiss
import numpy as np

import faissight

# 20k vectors in 64-d, drawn around 50 gaussian centres (a stand-in for embeddings).
rng = np.random.default_rng(0)
centres = rng.normal(scale=4.0, size=(50, 64)).astype("float32")
labels = rng.integers(0, 50, size=20_000)
xb = (centres[labels] + rng.normal(size=(20_000, 64))).astype("float32")

index = faiss.IndexIVFFlat(faiss.IndexFlatL2(64), 64, 128)
index.train(xb)
index.add(xb)
index.ntotal

Metadata is optional; anything with an `id` column works (list of dicts, DataFrame, `.jsonl`).

In [ ]:
chunks = [
    {"id": i, "text": f"passage {i} about topic {labels[i]}", "source": f"doc_{labels[i]}.md"}
    for i in range(len(xb))
]

## Launch the UI

Displaying the viewer embeds the UI below. Pass `vectors` for exact ground truth; without
them faissight reconstructs vectors from the index.

In [ ]:
viewer = faissight.launch(index, vectors=xb, metadata=chunks)
viewer

## The same answers from Python

`viewer.session` is the state behind the UI. Query a stored vector at `nprobe=1` and see
which true neighbours were missed, and why.

In [ ]:
report = viewer.session.query(id=42, k=10, nprobe=1)
trace = report.ivf_trace
print(f"recall@10 = {report.recall:.2f}")
print(f"all true neighbours need nprobe >= {trace.min_nprobe_for_all}")
for n in trace.neighbours:
    print(f"  id {n.id:>6}  list {n.list_no:>3}  probe rank {n.probe_rank:>3}  {n.reason.value}")

## Tune nprobe

Sweep nprobe over 100 sampled queries and pick the cheapest value reaching 95% recall.

In [ ]:
_, job = viewer.session.sweep_job(n_queries=100)
job.wait()
result = job.result
for p in result.points:
    print(f"nprobe={p.value:>4}  recall={p.recall:.3f}  {p.latency_mean_ms:.3f} ms/query")
best = result.recommend(0.95)
print("recommended nprobe:", best.value)

In [ ]:
viewer.stop()